# LoRA and QLoRA Mechanics

Full fine-tuning updates all model weights. [Parameter-efficient fine-tuning (PEFT)](https://huggingface.co/docs/peft/index) methods such as LoRA update a much smaller set while the pretrained base model stays frozen. This notebook makes that difference measurable before training.


## Learning Objectives

By the end of this notebook, you should be able to:

- Count trainable and frozen parameters before and after adding LoRA.
- Explain how adapter rank, scaling, dropout, and target modules affect adaptation.
- Compare full fine-tuning, LoRA, and QLoRA under different memory constraints.


## How LoRA Works

LoRA represents a weight update as two small matrices, commonly written as $\Delta W = BA$. If the original layer has shape $d_{out} \times d_{in}$ and the adapter rank is $r$, LoRA learns roughly $r(d_{in} + d_{out})$ values instead of $d_{in}d_{out}$ values. QLoRA adds quantized storage for the frozen base model while the LoRA adapters remain trainable.


<div align="center">
  <img src="assets/02-lora-diagram.png" alt="LoRA adapter diagram" width="700" style="max-width: 100%; height: auto;" />
  <p><em>LoRA during and after training: the frozen weights <code>W</code> plus a trainable low-rank update <code>BA</code>, which can be merged into <code>W</code> afterwards (this repository keeps the adapter separate instead). Source: <a href="https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/lora_diagram.png" target="_blank" rel="noopener noreferrer">Hugging Face PEFT LoRA diagram</a></em></p>
</div>


In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
MODEL_REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"
DEVICE = torch.device("cpu")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

## Count Parameters

Counting trainable parameters turns adapter training from an abstract concept into something you can inspect. The same check is useful in real projects before launching expensive training jobs.

Parameter count is only one part of memory use. Training also stores gradients, activations, and optimizer state. LoRA reduces gradient and optimizer-state memory because most base parameters stay frozen, but activations still depend on sequence length, batch size, and model architecture.


In [ ]:
def parameter_summary(model_to_count: torch.nn.Module) -> dict[str, float]:
    # Count all tensor elements, then count only tensors that require gradients.
    total = sum(parameter.numel() for parameter in model_to_count.parameters())
    trainable = sum(
        parameter.numel()
        for parameter in model_to_count.parameters()
        if parameter.requires_grad
    )
    return {
        "trainable": trainable,
        "total": total,
        "trainable_percent": round(100 * trainable / total, 4),
    }


parameter_summary(model)

### Exercise: Interpret Full Fine-Tuning Cost

Explain why full fine-tuning becomes expensive for large models.


<details>
<summary>Show Answer</summary>

Full fine-tuning is expensive because gradients and optimizer state must be stored for all model parameters, so memory and compute scale with the full model size.

</details>


## Add a LoRA Adapter

SmolLM2 uses separate query and value projections inside each attention block. We target `q_proj` and `v_proj`: the frozen projections still perform the original computation, while LoRA learns small task-specific updates.

The main adapter choices are:

- `r`: rank, which controls adapter capacity and parameter count;
- `lora_alpha`: scaling applied to the learned update;
- `lora_dropout`: regularization applied on the adapter path during training;
- `target_modules`: the layer types that receive adapters.

The learned update is scaled as $\frac{\alpha}{r} BA$, so `lora_alpha=16` with `r=8` doubles the adapter's contribution, and raising `r` without raising `lora_alpha` shrinks that scaling factor.


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

lora_model = get_peft_model(model, lora_config)
parameter_summary(lora_model)

### Exercise: Inspect Adapter Targets

List the first four module names ending in `q_proj` or `v_proj`.


In [ ]:
# @TODO: Inspect named modules and keep query/value projection names.


<details>
<summary>Show Answer</summary>

```python
adapter_target_names = [
    name
    for name, _ in lora_model.named_modules()
    if name.endswith(("q_proj", "v_proj"))
]
adapter_target_names[:4]
```

</details>


### Exercise: Compute Frozen Parameters

Use the summary dictionary to compute how many parameters are frozen after adding LoRA.


In [ ]:
# @TODO: Compute frozen parameter count from `parameter_summary(lora_model)`.


<details>
<summary>Show Answer</summary>

```python
lora_counts = parameter_summary(lora_model)
frozen_parameters = lora_counts["total"] - lora_counts["trainable"]
frozen_parameters
```

</details>


## Where QLoRA Fits

**Quantization** represents weights with fewer bits. Values are mapped to a limited set of levels and reconstructed approximately during computation. Uniform integer quantization uses a scale and sometimes a zero-point; QLoRA typically uses NF4, a non-uniform 4-bit codebook. For SmolLM2-135M, the weights alone take about 540 MB in 32-bit floats, 270 MB in 16-bit, and roughly 70 MB at 4 bits, before quantization metadata. The recovered weights only approximate the originals.

QLoRA combines quantization and LoRA. The base model is stored in low precision while adapter weights remain trainable, reducing memory use while adding quantization-specific complexity.

| Approach | Base weights | Updated parameters | Typical reason to choose it |
| --- | --- | --- | --- |
| Full fine-tuning | Floating point (often 16-bit) | All weights | Maximum flexibility when compute and data are abundant. |
| LoRA | Usually full or half precision | Low-rank adapters | Lower training cost and small portable checkpoints. |
| QLoRA | Usually 4-bit quantized | Low-rank adapters | Fit a larger base model into limited accelerator memory. |

Quantization is not automatically better. It can introduce hardware constraints, numerical trade-offs, and extra configuration. For this CPU-friendly repository, ordinary LoRA keeps the runnable path simpler; QLoRA is taught as the next scaling step.


<div align="center">
  <img src="assets/02-dequant.png" alt="Dequantization visual" width="700" style="max-width: 100%; height: auto;" />
  <p><em>Uniform int8 dequantization: subtract the zero-point <code>z</code> from stored values <code>q</code>, then multiply by the scale <code>s</code> to recover approximate FP32 weights <code>r</code>. This illustrates reconstruction, not QLoRA's NF4 codebook. Source: <a href="https://huggingface.co/datasets/huggingface/documentation-images/blob/main/transformers/dequant.png" target="_blank" rel="noopener noreferrer">Hugging Face documentation images</a></em></p>
</div>


### Exercise: Choose LoRA or QLoRA

For a 7B parameter model on a limited-memory GPU, which approach is more likely to fit: full fine-tuning, LoRA, or QLoRA?


<details>
<summary>Show Answer</summary>

QLoRA is usually most memory-efficient because the base model can be loaded in low precision while only adapter parameters are trained. LoRA also reduces memory, but typically less than QLoRA.

</details>


## Further Reading

- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) - Hu et al., 2021
- [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314) - Dettmers et al., 2023
- [LoRA Developer Guide](https://huggingface.co/docs/peft/developer_guides/lora)
- [Quantization Overview](https://huggingface.co/docs/transformers/quantization/overview)
- [Prompt-Based Methods: Prompt Tuning, Prefix Tuning, and P-Tuning](https://huggingface.co/docs/peft/main/conceptual_guides/prompting)